# 7차시 · 보상으로 배우는 AI (강화학습 · Q-러닝)

**탐구 질문**: AI에게 "점수"만 주면 우리가 원하는 대로 행동할까? 점수 규칙을 잘못 만들면 무슨 일이 생길까?

핵심 한 줄: `Q[s][a] += ALPHA * (r + GAMMA * max(Q[s2]) - Q[s][a])`

> 🔧 **규칙**: 맨 위 대문자 변수만, 한 번에 **하나씩** 바꾸고, 실행 **전에** 결과를 예측해 적어 보세요.

## 코드 실험실
아래 셀을 실행(Shift+Enter)해 보세요.

In [ ]:
# 7차시 · 보상으로 배우는 AI (Q-러닝으로 미로 탈출)
# 🔧 대문자 변수만 바꿔 가며 실험해 보세요. 한 번에 '하나만'!
EPISODES = 300      # 연습 판 수
ALPHA = 0.5         # 학습률: 새 경험을 얼마나 크게 반영할까? (1차시의 학습률과 같은 역할)
GAMMA = 0.9         # 할인율: 미래 보상을 얼마나 중요하게 볼까? (0 = 지금만, 1 = 먼 미래까지)
EPSILON = 0.1       # 탐험률: 일부러 아무 길이나 가 볼 확률
STEP_REWARD = -1    # 한 칸 움직일 때마다 받는 점수 (이동 비용)
TRAP_REWARD = -10   # 함정(X)에 빠지면 받는 점수 (그 판은 끝)
GOAL_REWARD = 10    # 도착(G)하면 받는 점수 (그 판은 끝)
SEED = 1

import random
import matplotlib.pyplot as plt

MAP = [
    "....G",
    ".#X..",
    "...X.",
    ".X...",
    "S....",
]
ROWS, COLS = len(MAP), len(MAP[0])
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]      # 위, 아래, 왼쪽, 오른쪽
ARROWS = ["↑", "↓", "←", "→"]
start = next((r, c) for r in range(ROWS) for c in range(COLS) if MAP[r][c] == "S")

def step(state, a):
    """한 칸 움직이고 (다음 위치, 보상, 끝났나?)를 돌려준다"""
    r, c = state[0] + ACTIONS[a][0], state[1] + ACTIONS[a][1]
    if not (0 <= r < ROWS and 0 <= c < COLS) or MAP[r][c] == "#":
        r, c = state                                 # 벽이면 제자리
    cell = MAP[r][c]
    if cell == "G": return (r, c), GOAL_REWARD, True
    if cell == "X": return (r, c), TRAP_REWARD, True
    return (r, c), STEP_REWARD, False

# Q표: 칸마다, 방향마다 "이 방향이 앞으로 얼마나 좋은가" 점수 (처음엔 모두 0)
Q = {(r, c): [0.0, 0.0, 0.0, 0.0] for r in range(ROWS) for c in range(COLS)}

def best_action(state):
    q = Q[state]
    m = max(q)
    return random.choice([a for a in range(4) if q[a] == m])  # 동점이면 무작위

random.seed(SEED)
scores = []
for ep in range(EPISODES):
    s, total = start, 0
    for t in range(100):                              # 한 판에 최대 100걸음
        if random.random() < EPSILON:
            a = random.randrange(4)                   # 탐험: 아무 방향
        else:
            a = best_action(s)                        # 이용: 지금까지 가장 좋았던 방향
        s2, r, done = step(s, a)
        future = 0 if done else max(Q[s2])
        Q[s][a] += ALPHA * (r + GAMMA * future - Q[s][a])   # ⭐ Q-러닝 핵심 한 줄
        s, total = s2, total + r
        if done:
            break
    scores.append(total)

# 다 배운 뒤 탐험 없이 한 번 가 보기
s, path, total, result = start, [start], 0, "30걸음 안에 도착 못 함"
for t in range(30):
    s, r, done = step(s, best_action(s))
    path.append(s); total += r
    if done:
        result = "도착 성공!" if MAP[s[0]][s[1]] == "G" else "함정에 빠짐"
        break

print("[배운 방향 지도]  (각 칸에서 Q가 가장 큰 방향)")
for r in range(ROWS):
    row = ""
    for c in range(COLS):
        ch = MAP[r][c]
        row += f" {ch} " if ch in "SG#X" else (" * " if (r, c) in path else f" {ARROWS[Q[(r, c)].index(max(Q[(r, c)]))]} ")
    print(row)
print("(* = 마지막에 실제로 지나간 길)")

plt.figure(figsize=(7, 3))
plt.plot(scores, alpha=0.35, label="score per episode")
avg = [sum(scores[max(0, i - 19): i + 1]) / len(scores[max(0, i - 19): i + 1]) for i in range(len(scores))]
plt.plot(avg, linewidth=2, label="moving average (20)")
plt.xlabel("episode"); plt.ylabel("total reward"); plt.legend(); plt.grid(alpha=.3)
plt.title("Learning curve"); plt.tight_layout(); plt.show()

print(f"\n마지막 50판 평균 점수: {sum(scores[-50:]) / 50:.1f}")
print(f"결과: {result} · {len(path) - 1}걸음 · 점수 {total}")

## 탐구 미션
각 미션마다 위 셀의 변수를 바꿔 다시 실행하고, 이 칸을 더블클릭해 예측·결과·설명을 적어 보세요.

### M1. 연습량과 학습률
`EPISODES`를 `300 → 20`, 다시 300으로 돌리고 `ALPHA`를 `0.5 → 0.01`

- **예측**: 
- **결과**: 
- **설명**: 

### M2. 탐험의 대가
`EPSILON`을 `0.1 → 0.5`. 최종 결과와 마지막 50판 평균 점수를 따로 비교

- **예측**: 
- **결과**: 
- **설명**: 

### M3. 미래를 보지 않는 AI
`GAMMA`를 `0.9 → 0.5 → 0.0`

- **예측**: 
- **결과**: 
- **설명**: 

### M4. 보상을 잘못 설계하면
`STEP_REWARD`를 `-1 → -2 → -5`

- **예측**: 
- **결과**: 
- **설명**: 

### M5. 코드 추가: 함정 카운터
연습하는 동안 함정에 빠진 횟수를 세는 변수 `falls`를 만들어 마지막에 출력해 보세요. 그다음 `EPSILON` 0.1과 0.5에서 값을 비교!

- **예측**: 
- **결과**: 
- **설명**: 

## 나의 결론
오늘의 탐구 질문에 대한 나의 결론 (3문장 이상):